# Sim2Real AI Calibration Pipeline

이 노트북은 아래 3단계를 한 번에 검증합니다.

1. **AI 필요성 확인**: 전통(Persistence, LinearRidge) vs AI(MLP) 성능 비교
2. **시뮬레이션 보정**: real 유사도 + validation NMSE를 함께 최적화해 노이즈 파라미터 선택
3. **보정된 sim 기반 채널 추정**: 보정 전/후 sim으로 학습했을 때 real에서 성능이 개선되는지 확인


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

module_path = Path('/home/mh/kmh/sionna-rt/jinsup/sim2real_ai_vs_traditional.py')
spec = importlib.util.spec_from_file_location('bench_mod', module_path)
mod = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(mod)

print(f'Loaded module: {module_path}')


In [ ]:
# ===== 파라미터 =====
sim_path = '/home/mh/kmh/sionna-rt/jinsup/cir_pdp_exports'
real_path = '/data/kmh/sionna-rt/mh/workspace/data_analyze/data/CIR.mat'  # 비우면 auto detect
real_key = 'CIR'

window = 8
horizon = 1
max_delay = 128
calib_ratio = 0.25

ridge_alpha = 1.0
mlp_hidden = 128
mlp_epochs = 80
mlp_fine_tune_epochs = 20
mlp_batch_size = 32
mlp_lr = 1e-3
mlp_input_clip = 8.0
seed = 1234

lambda_similarity = 0.6  # J(theta)에서 유사도 비중

fig_dir = Path('/home/mh/kmh/sionna-rt/jinsup/realVSsim/ai_calibration_pipeline')
fig_dir.mkdir(parents=True, exist_ok=True)
fig_dir


In [ ]:
# ===== 데이터 로드 =====
resolved_real_path = mod._resolve_real_path(real_path)
if not resolved_real_path:
    raise FileNotFoundError('real_path를 찾지 못했습니다. real_path를 직접 지정하세요.')

real_c, real_key_used = mod.load_cir(Path(resolved_real_path), real_key, 'auto')
sim_clean, _ = mod.load_cir(Path(sim_path), 'h_cir_clean', 'auto')
sim_noisy, _ = mod.load_cir(Path(sim_path), 'h_cir_noisy', 'auto')

common_d = min(max_delay, real_c.shape[1], sim_clean.shape[1], sim_noisy.shape[1])
real_cd = real_c[:, :common_d]
sim_clean_d = sim_clean[:, :common_d]
sim_noisy_d = sim_noisy[:, :common_d]

print('real:', real_cd.shape, 'key=', real_key_used)
print('sim_clean:', sim_clean_d.shape)
print('sim_noisy:', sim_noisy_d.shape)


In [ ]:
# ===== 유틸 함수 =====
def inject_noise_like_measurement(h_clean, awgn_rel_db=-45.0, phase_std_deg=0.0, amp_jitter_std_db=0.0, seed=1234):
    rng = np.random.default_rng(seed)
    h = np.array(h_clean, dtype=np.complex128, copy=True)

    if amp_jitter_std_db > 0.0:
        amp_db = rng.normal(0.0, amp_jitter_std_db, size=(h.shape[0], 1))
        h = h * (10.0 ** (amp_db / 20.0))

    if phase_std_deg > 0.0:
        phi = rng.normal(0.0, np.deg2rad(phase_std_deg), size=(h.shape[0], 1))
        h = h * np.exp(1j * phi)

    peak_p = np.max(np.abs(h) ** 2, axis=1, keepdims=True)
    noise_p = np.maximum(peak_p, 1e-30) * (10.0 ** (awgn_rel_db / 10.0))
    sigma = np.sqrt(noise_p / 2.0)
    n_re = rng.normal(0.0, 1.0, size=h.shape) * sigma
    n_im = rng.normal(0.0, 1.0, size=h.shape) * sigma
    return h + (n_re + 1j * n_im)


def split_real_windows(real_complex, window, horizon, calib_ratio=0.25):
    rf = mod.complex_to_features(real_complex)
    x_real, y_real = mod.make_windows(rf, window, horizon)

    n = x_real.shape[0]
    n_tune = int(round(n * calib_ratio))
    n_tune = max(1, min(n - 2, n_tune))

    x_tune, y_tune = x_real[:n_tune], y_real[:n_tune]
    x_rest, y_rest = x_real[n_tune:], y_real[n_tune:]
    n_val = max(1, len(x_rest)//2)
    x_val, y_val = x_rest[:n_val], y_rest[:n_val]
    x_test, y_test = x_rest[n_val:], y_rest[n_val:]

    return x_tune, y_tune, x_val, y_val, x_test, y_test


def eval_on_real(sim_complex, scenario_name, seed=1234):
    sim_f = mod.complex_to_features(sim_complex)
    x_sim, y_sim = mod.make_windows(sim_f, window, horizon)

    x_tune, y_tune, x_val, y_val, x_test, y_test = split_real_windows(real_cd, window, horizon, calib_ratio)

    # val 기준 비교
    y_val_true_c = mod.features_to_complex(y_val)
    fdim = y_val.shape[1]

    y_persist_val = x_val.reshape(-1, window, fdim)[:, -1, :]
    m_persist = mod.eval_prediction(y_val_true_c, mod.features_to_complex(y_persist_val))

    y_ridge_val = mod.train_ridge_predict(
        x_sim=x_sim, y_sim=y_sim,
        x_cal=x_tune, y_cal=y_tune,
        x_eval=x_val, alpha=ridge_alpha,
    )
    m_ridge = mod.eval_prediction(y_val_true_c, mod.features_to_complex(y_ridge_val))

    y_mlp_val = mod.train_mlp_predict(
        x_sim=x_sim, y_sim=y_sim,
        x_cal=x_tune, y_cal=y_tune,
        x_eval=x_val,
        hidden=mlp_hidden,
        epochs=mlp_epochs,
        fine_tune_epochs=mlp_fine_tune_epochs,
        batch_size=mlp_batch_size,
        lr=mlp_lr,
        clip=mlp_input_clip,
        seed=seed,
    )
    m_mlp = mod.eval_prediction(y_val_true_c, mod.features_to_complex(y_mlp_val))

    # test에서 Ridge만 확인 (파라미터 선택 후 일반화 체크)
    y_test_true_c = mod.features_to_complex(y_test)
    y_ridge_test = mod.train_ridge_predict(
        x_sim=x_sim, y_sim=y_sim,
        x_cal=x_tune, y_cal=y_tune,
        x_eval=x_test, alpha=ridge_alpha,
    )
    m_ridge_test = mod.eval_prediction(y_test_true_c, mod.features_to_complex(y_ridge_test))

    rows = []
    for model, mm in [('Persistence', m_persist), ('LinearRidge', m_ridge), ('AI-MLP', m_mlp)]:
        rows.append({
            'scenario': scenario_name,
            'split': 'val',
            'model': model,
            'mse': mm['mse'],
            'mae': mm['mae'],
            'nmse': mm['nmse'],
            'nmse_improve_vs_persistence_pct': 0.0 if model=='Persistence' else 100.0*(1.0 - mm['nmse']/(m_persist['nmse']+1e-12)),
        })

    rows.append({
        'scenario': scenario_name,
        'split': 'test',
        'model': 'LinearRidge',
        'mse': m_ridge_test['mse'],
        'mae': m_ridge_test['mae'],
        'nmse': m_ridge_test['nmse'],
        'nmse_improve_vs_persistence_pct': np.nan,
    })

    return rows


## Step A. AI 필요성 확인

`sim_clean`, `sim_noisy` 각각에서 전통 방식 vs AI를 비교합니다.


In [ ]:
rows = []
rows += eval_on_real(sim_clean_d, 'sim_clean', seed=seed)
rows += eval_on_real(sim_noisy_d, 'sim_noisy', seed=seed)

ai_need_df = pd.DataFrame(rows)
display(ai_need_df.sort_values(['scenario', 'split', 'model']))

# val split NMSE bar
plot_df = ai_need_df[ai_need_df['split']=='val'].copy()
models = ['Persistence', 'LinearRidge', 'AI-MLP']
scenarios = sorted(plot_df['scenario'].unique())

x = np.arange(len(models))
w = 0.8 / max(1, len(scenarios))
plt.figure(figsize=(10,4))
for i, sc in enumerate(scenarios):
    vals = [float(plot_df[(plot_df['scenario']==sc)&(plot_df['model']==m)]['nmse'].iloc[0]) for m in models]
    shift = (i - (len(scenarios)-1)/2.0) * w
    plt.bar(x + shift, vals, width=w, label=sc)

plt.xticks(x, models)
plt.ylabel('Validation NMSE (lower better)')
plt.title('Step A: AI Necessity Check')
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / 'A1_ai_necessity_nmse.png', dpi=150)
plt.show()


## Step B. 노이즈 파라미터 자동 선택

목적함수: `J(θ) = λ * Similarity + (1-λ) * Prediction`

- Similarity = `overall_similarity_score / 100`
- Prediction = `1/(1+val_nmse_ridge)`


In [ ]:
# similarity helpers (same spirit as previous notebook)
def safe_corr(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(np.corrcoef(a,b)[0,1])


def cosine_sim(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    return float(np.dot(a,b)/(na*nb))


def js_divergence_from_hist(x, y, bins=200):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)
    mn = float(min(np.min(x), np.min(y)))
    mx = float(max(np.max(x), np.max(y)))
    if mx - mn < 1e-12:
        return 0.0
    px, _ = np.histogram(x, bins=bins, range=(mn,mx), density=True)
    py, _ = np.histogram(y, bins=bins, range=(mn,mx), density=True)
    px = (px + 1e-12); py = (py + 1e-12)
    px = px/np.sum(px); py = py/np.sum(py)
    m = 0.5*(px+py)
    return float(0.5*np.sum(px*np.log(px/m)) + 0.5*np.sum(py*np.log(py/m)))


def framewise_mag_corr_mean(real_c, sim_c):
    n = min(real_c.shape[0], sim_c.shape[0])
    vals = []
    for t in range(n):
        vals.append(safe_corr(np.abs(real_c[t]), np.abs(sim_c[t])))
    return float(np.mean(vals))


def similarity_score(sim_arr, real_arr):
    base = mod.similarity_metrics(sim_arr, real_arr)
    real_pdp = np.mean(np.abs(real_arr)**2, axis=0)
    sim_pdp = np.mean(np.abs(sim_arr)**2, axis=0)
    pdp_cos = cosine_sim(sim_pdp, real_pdp)
    frame_corr = framewise_mag_corr_mean(real_arr, sim_arr)
    mag_js = js_divergence_from_hist(np.abs(sim_arr).ravel(), np.abs(real_arr).ravel())

    c1 = np.clip((pdp_cos + 1.0)/2.0, 0.0, 1.0)
    c2 = np.clip((base['energy_corr'] + 1.0)/2.0, 0.0, 1.0)
    c3 = np.clip((frame_corr + 1.0)/2.0, 0.0, 1.0)
    c4 = 1.0/(1.0 + max(base['pdp_nmse'], 0.0))
    c5 = 1.0/(1.0 + max(base['energy_nmse'], 0.0))
    c6 = 1.0/(1.0 + max(mag_js, 0.0))

    score = 100.0 * np.mean([c1,c2,c3,c4,c5,c6])
    return score


awgn_grid = [-55, -50, -45, -40, -35]
phase_grid = [0.0, 1.0, 2.0, 5.0]
amp_grid = [0.0, 0.5, 1.0]

x_tune, y_tune, x_val, y_val, x_test, y_test = split_real_windows(real_cd, window, horizon, calib_ratio)
y_val_c = mod.features_to_complex(y_val)

rows = []
for awgn in awgn_grid:
    for ph in phase_grid:
        for amp in amp_grid:
            sim_syn = inject_noise_like_measurement(sim_clean_d, awgn_rel_db=awgn, phase_std_deg=ph, amp_jitter_std_db=amp, seed=seed)

            # similarity term
            sim_term = similarity_score(sim_syn, real_cd) / 100.0

            # prediction term (ridge val)
            x_sim, y_sim = mod.make_windows(mod.complex_to_features(sim_syn), window, horizon)
            y_val_pred = mod.train_ridge_predict(
                x_sim=x_sim, y_sim=y_sim,
                x_cal=x_tune, y_cal=y_tune,
                x_eval=x_val, alpha=ridge_alpha,
            )
            m_val = mod.eval_prediction(y_val_c, mod.features_to_complex(y_val_pred))
            val_nmse = float(m_val['nmse'])
            pred_term = 1.0/(1.0 + max(val_nmse, 0.0))

            J = lambda_similarity * sim_term + (1.0 - lambda_similarity) * pred_term

            rows.append({
                'awgn_rel_db': awgn,
                'phase_std_deg': ph,
                'amp_jitter_std_db': amp,
                'similarity_score': sim_term*100.0,
                'val_nmse_ridge': val_nmse,
                'J_objective': J,
            })

noise_select_df = pd.DataFrame(rows).sort_values('J_objective', ascending=False).reset_index(drop=True)
best_theta = noise_select_df.iloc[0]

print('[best theta]')
display(best_theta)
display(noise_select_df.head(10))

# visualize top-10 by J
top = noise_select_df.head(10)
labels = [f"N{r.awgn_rel_db}/P{r.phase_std_deg}/A{r.amp_jitter_std_db}" for _, r in top.iterrows()]

x = np.arange(len(top))
fig, ax1 = plt.subplots(figsize=(11,4))
ax1.bar(x, top['J_objective'], color='#4C78A8', alpha=0.85)
ax1.set_ylabel('J objective (higher better)')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=30, ha='right')
ax1.set_title('Step B: Top-10 Noise Params by J(theta)')

ax2 = ax1.twinx()
ax2.plot(x, top['val_nmse_ridge'].values, color='#F58518', marker='o')
ax2.set_ylabel('Validation NMSE (lower better)')

fig.tight_layout()
plt.savefig(fig_dir / 'B1_noise_selection_objective.png', dpi=150)
plt.show()


## Step C. 보정 sim 기반 채널 추정 성능 확인

`sim_tuned_from_clean(best_theta)`를 추가해, 보정 전/후가 real에서 얼마나 차이 나는지 비교합니다.


In [ ]:
# build tuned scenario from best theta
sim_tuned = inject_noise_like_measurement(
    sim_clean_d,
    awgn_rel_db=float(best_theta['awgn_rel_db']),
    phase_std_deg=float(best_theta['phase_std_deg']),
    amp_jitter_std_db=float(best_theta['amp_jitter_std_db']),
    seed=seed,
)

rows = []
rows += eval_on_real(sim_clean_d, 'sim_clean', seed=seed)
rows += eval_on_real(sim_noisy_d, 'sim_noisy', seed=seed)
rows += eval_on_real(sim_tuned, 'sim_tuned_from_clean', seed=seed)

final_df = pd.DataFrame(rows)
display(final_df.sort_values(['split','scenario','model']))

# val split bar chart
plot_df = final_df[final_df['split']=='val'].copy()
models = ['Persistence', 'LinearRidge', 'AI-MLP']
scenarios = ['sim_clean', 'sim_noisy', 'sim_tuned_from_clean']

x = np.arange(len(models))
w = 0.26
plt.figure(figsize=(10,4))
for i, sc in enumerate(scenarios):
    vals = [float(plot_df[(plot_df['scenario']==sc)&(plot_df['model']==m)]['nmse'].iloc[0]) for m in models]
    plt.bar(x + (i-1)*w, vals, width=w, label=sc)

plt.xticks(x, models)
plt.ylabel('Validation NMSE (lower better)')
plt.title('Step C: Before/After Calibration on Real Validation')
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / 'C1_before_after_calibration_nmse.png', dpi=150)
plt.show()

print('
Best theta used for tuned sim:')
print(best_theta[['awgn_rel_db','phase_std_deg','amp_jitter_std_db','similarity_score','val_nmse_ridge','J_objective']])


In [ ]:
# ===== 저장 =====
ai_need_csv = fig_dir / 'A_ai_necessity_metrics.csv'
noise_sel_csv = fig_dir / 'B_noise_selection_metrics.csv'
final_csv = fig_dir / 'C_final_comparison_metrics.csv'

if 'ai_need_df' in globals():
    ai_need_df.to_csv(ai_need_csv, index=False)
if 'noise_select_df' in globals():
    noise_select_df.to_csv(noise_sel_csv, index=False)
if 'final_df' in globals():
    final_df.to_csv(final_csv, index=False)

print('saved:', ai_need_csv)
print('saved:', noise_sel_csv)
print('saved:', final_csv)
print('saved images in:', fig_dir)


## 해석 가이드

- Step A에서 `AI-MLP`가 전통 대비 우세하면 AI 필요성 근거가 됩니다.
- Step B에서 `best_theta`는 유사도와 예측 성능을 동시에 고려한 노이즈 보정 파라미터입니다.
- Step C에서 `sim_tuned_from_clean`이 `sim_clean`보다 NMSE가 낮으면,
  **보정된 시뮬레이션으로 학습한 모델이 real에서 더 잘 동작**한다는 근거가 됩니다.
